# Spark Notes

####1. How to create a Spark Session

In [0]:
# import spark session builder api
# from pyspark.sql import SparkSession

# get or create either gets current or creates a new spark session
# spark_session = SparkSession.builder.getOrCreate()

#####Get the spark version

In [0]:
# get the current spark version number
spark_session.version

In [0]:
spark.version
# these are the same version because they are the same spark session

##### Show a table using dataframe

In [0]:
df = spark.table("sales_data.sales_db.customers_master")
# df.show()
df.display()

* Tables are stored
* Dataframe are in-memory two-dimensional labeled structures

#####Spark dataframe support APIs to:
* select
* filter
* join
* aggregate
* save
#####Spark dataframe advantages for big data:
* parallel processing
  * because of distributed computing with clusters
* in-memory processing
* schema flexibility
* fault tolerance
  * might not be possible in some tables

##### Perform the following tasks:
1. Read data file
2. Load to a table
3. Verify records

In [0]:
df = spark.read.csv("/Volumes/sales_data/sales_db/sales_raw/store_locations.csv", header=True, inferSchema=True)
# df.display()

# df.count()

# df.printSchema()

#transform the opening_year column to a date
# from pyspark.sql.functions import to_date
# df = df.withColumn("opening_year", to_date(df.opening_year, "yyyy"))
# df.printSchema()

df.write.mode("overwrite").saveAsTable("sales_data.sales_db.store_locations")

In [0]:
%sql
USE CATALOG sales_data;
SELECT * FROM sales_db.store_locations;

In [0]:
%sql
USE CATALOG sales_data;
SELECT COUNT(*) FROM sales_db.store_locations;

In [0]:
df_count = df.count()
tbl_count = spark.sql("SELECT COUNT(*) FROM sales_db.store_locations").collect()[0][0]

# print(df_count)
# print(tbl_count)

if df_count == tbl_count:
    print(f"Success: {df_count} of {tbl_count} rows written to table")
else:
    print("Failure: Rows were lost during the write")

#### Dataframe structure:
1. Transformations
2. Actions
3. Dataframewriter
4. Auxiliary methods

Display the table conents for review before creating query

In [0]:
%sql
SELECT * FROM sales_db.sales_raw_oltp;

##### Create sql query to find which store sold the most items

In [0]:
%sql
USE CATALOG sales_data;

CREATE OR REPLACE VIEW sales_db.vw_top_10_stores_by_units_sold AS
SELECT store_id, SUM(quantity) AS total_sold
FROM sales_db.sales_raw_oltp
GROUP BY store_id
ORDER BY total_sold DESC
LIMIT 10;

##### Create a sql query join showing the top 10 most popular items sold

In [0]:
%sql
USE CATALOG sales_data;

CREATE OR REPLACE VIEW sales_data.sales_db.vw_top_10_products_by_sales AS
SELECT s.product_id, p.product_name, SUM(s.quantity) AS total_sales
FROM sales_db.sales_raw_oltp s
JOIN sales_db.products_catalog p ON s.product_id = p.product_id
GROUP BY s.product_id, p.product_name
ORDER BY total_sales DESC
LIMIT 10;

##### query the new views

In [0]:
%sql
USE CATALOG sales_data;

SELECT * FROM sales_db.vw_top_10_products;

In [0]:
%sql
USE CATALOG sales_data;

SELECT * FROM sales_db.vw_top_10_stores_by_units_sold;

##### View query using dataframe

In [0]:
# SELECT store_id, SUM(quantity) AS total_sold
# FROM sales_db.sales_raw_oltp
# GROUP BY store_id
# ORDER BY total_sold DESC
# LIMIT 10;

from pyspark.sql.functions import col, sum

spark.sql("USE CATALOG sales_data")

df = spark.read.table("sales_data.sales_db.sales_raw_oltp")

df = df.groupBy("store_id").agg(sum("quantity").alias("total_sold")).orderBy(col("total_sold").desc()).limit(10)
display(df)

##### View query using dataframe

In [0]:
# SELECT s.product_id, p.product_name, SUM(s.quantity) AS popular_items
# FROM sales_db.sales_raw_oltp s
# JOIN sales_db.products_catalog p ON s.product_id = p.product_id
# GROUP BY s.product_id, p.product_name
# ORDER BY popular_items DESC
# LIMIT 10;

# adjust for differences
from pyspark.sql.functions import col, sum

spark.sql("USE CATALOG sales_data")

df1 = spark.read.table("sales_data.sales_db.sales_raw_oltp").alias("s")
df2 = spark.read.table("sales_data.sales_db.products_catalog").alias("p")

df_join = df1.join(df2, df1.product_id == df2.product_id)

df_join = df_join.groupBy("s.product_id", "p.product_name").agg(sum("quantity").alias("popular_items")).orderBy(col("popular_items").desc()).limit(10)
display(df_join)

##### Create views for each customer loyalty tier type to quickly show a list of customers at each tier

1. Bronze

In [0]:
%sql

USE CATALOG sales_data;

CREATE OR REPLACE VIEW sales_data.sales_db.vw_bronze_customers AS
SELECT customer_id, first_name, last_name, email, phone, loyalty_tier
FROM sales_db.customers_master
WHERE loyalty_tier = "Bronze";

2. Silver

In [0]:
%sql

USE CATALOG sales_data;

CREATE OR REPLACE VIEW sales_data.sales_db.vw_silver_customers AS
SELECT customer_id, first_name, last_name, email, phone, loyalty_tier
FROM sales_db.customers_master
WHERE loyalty_tier = "Silver";

3. Gold

In [0]:
%sql

USE CATALOG sales_data;

CREATE OR REPLACE VIEW sales_data.sales_db.vw_gold_customers AS
SELECT customer_id, first_name, last_name, email, phone, loyalty_tier
FROM sales_db.customers_master
WHERE loyalty_tier = "Gold";

4. Platinum

In [0]:
%sql

USE CATALOG sales_data;

CREATE OR REPLACE VIEW sales_data.sales_db.vw_platinum_customers AS
SELECT customer_id, first_name, last_name, email, phone, loyalty_tier
FROM sales_db.customers_master
WHERE loyalty_tier = "Platinum";

In [0]:
%sql
-- join on product_id with sales_raw_oltp

USE CATALOG sales_data;

CREATE OR REPLACE VIEW sales_data.sales_db.vw_top_10_brands_by_sales AS
SELECT p.brand, COUNT(*) AS brand_sales
FROM sales_db.products_catalog p
JOIN sales_db.sales_raw_oltp s ON p.product_id = s.product_id
GROUP BY s.product_id, p.brand
ORDER BY brand_sales DESC
LIMIT 10;

In [0]:
%sql
USE CATALOG sales_data;

CREATE OR REPLACE VIEW sales_data.sales_db.vw_staff_count_by_store AS
SELECT l.store_name, COUNT(r.staff_id) AS staff_count
FROM sales_db.store_locations l
RIGHT JOIN sales_db.staff_records r ON l.store_id = r.store_id
GROUP BY l.store_name
ORDER BY staff_count DESC;

In [0]:
%sql
USE CATALOG sales_data;

CREATE OR REPLACE VIEW sales_data.sales_db.vw_store_count_by_region AS
SELECT region, COUNT(store_id) AS store_count
FROM sales_db.store_locations
GROUP BY region
ORDER BY store_count DESC;

In [0]:
df = spark.read.table("sales_data.sales_db.customers_master")

df.explain(mode="extended")

##### 5 different ways to create a dataframe

##### Schema on read

In [0]:
%sql
USE CATALOG sales_data;

SELECT * FROM sales_db.vw_top_10_products_by_sales;